# 03 — Arm 2: Fine-tuned BERT classifier

Dataset C, AMLH coursework. `question` -> WordPiece (`max_length=48`) -> encoder -> 906-way
linear classification head -> argmax. `answer` is never an input to this arm. Labels are encoded
against the full 906-class training universe so the label space matches Arm 1 and the frozen
test run. No test-set cell anywhere in this notebook. All logic lives in `src/amlh/arm2_bert.py`;
this notebook only calls it, displays results, and persists tables to `artefacts/` and figures to
`figures/`. It runs standalone after a kernel restart by loading `artefacts/split_fit.csv` and
`artefacts/split_val.csv`.

**`QUICK_SMOKE_TEST`** (next cell): this environment is CPU-only, and the real frozen run
targets a Colab T4. Set to `True` to subsample fit/val and cut epochs down to a fast
correctness check — every cell still executes real code and prints real numbers, but the
printed hyperparameters at the end are **not** valid to freeze into `config.py`. Set to `False`
(on Colab, GPU available) for the real run whose output gets hand-copied into `config.py`.

In [1]:
!git clone https://github.com/L1N-z/amlh-proj.git
%cd amlh-proj
!pip install -e ".[bert]" -q

Cloning into 'amlh-proj'...
remote: Enumerating objects: 125, done.
remote: Counting objects: 100% (125/125), done.
remote: Compressing objects: 100% (66/66), done.
remote: Total 125 (delta 49), reused 122 (delta 48), pack-reused 0 (from 0)
Receiving objects: 100% (125/125), 1016.57 KiB | 10.81 MiB/s, done.
Resolving deltas: 100% (49/49), done.
/content/amlh-proj
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for amlh (pyproject.toml) ... done


In [ ]:
from google.colab import files
uploaded = files.upload()  # upload a zip of data/ and artefacts/ made locally
!unzip -o arm2_colab_inputs.zip -d artefacts/

In [3]:
QUICK_SMOKE_TEST = True

In [4]:
import sys
sys.path.insert(0, "/content/amlh-proj/src")

In [ ]:
from pathlib import Path

try:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_DIR = Path("/content/drive/MyDrive/amlh_arm2")
    DRIVE_DIR.mkdir(parents=True, exist_ok=True)
    print(f"Drive mounted -- checkpoints will be saved to {DRIVE_DIR}")
except Exception as e:
    DRIVE_DIR = None
    print(f"Drive unavailable ({e}) -- checkpoints will not be saved this run")

In [5]:
import time

import matplotlib.pyplot as plt
import pandas as pd
import torch

from amlh import arm2_bert as ab
from amlh import evaluate
from amlh.config import ARTEFACTS_DIR, FIGURES_DIR, set_seed
from amlh.data import load_train

## 1. Load splits from `artefacts/`

In [ ]:
set_seed()

fit = pd.read_csv(ARTEFACTS_DIR / "split_fit.csv")
val = pd.read_csv(ARTEFACTS_DIR / "split_val.csv")

if QUICK_SMOKE_TEST:
    # Subsample for a fast CPU correctness check only -- never for the frozen run.
    fit = fit.groupby("disease", group_keys=False)[fit.columns].apply(lambda g: g.head(1)).sample(
        n=40, random_state=44
    ).reset_index(drop=True)
    val = val.sample(n=20, random_state=44).reset_index(drop=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none"
print(f"fit={len(fit)} ({fit.disease.nunique()} classes) | val={len(val)} ({val.disease.nunique()} classes)")
print(f"device={device} | gpu={gpu_name}")

if not QUICK_SMOKE_TEST:
    assert device.type == "cuda", "real run needs a T4 -- Runtime > Change runtime type"

## 2. Label encoding over the full 906-class universe

In [8]:
import os
import pandas as pd
from google.colab import files

os.makedirs("data", exist_ok=True)
# Check if the file is already there or if we need to upload
if not os.path.exists("patient_qa_classification_train.zip"):
    uploaded = files.upload()

# Unzip the correctly named file into the 'data' directory
!unzip -o patient_qa_classification_train.zip -d data/

# Verify it landed in the right place with the right shape
train_path = "data/patient_qa_classification_train.csv"
train_check = pd.read_csv(train_path)

assert {"question", "answer", "disease", "reference_url"} <= set(train_check.columns), train_check.columns
assert train_check["disease"].nunique() == 906, train_check["disease"].nunique()
print(f"{train_path} OK: {len(train_check)} rows, {train_check['disease'].nunique()} classes")

Saving patient_qa_classification_train.zip to patient_qa_classification_train.zip
Archive:  patient_qa_classification_train.zip
  inflating: data/patient_qa_classification_train.csv  
data/patient_qa_classification_train.csv OK: 8891 rows, 906 classes


In [ ]:
# Must be the full training CSV, not `fit` -- `fit` is a split (and under
# QUICK_SMOKE_TEST a 40-class subsample), so encoding labels from it would leave the output
# layer missing classes the frozen test run could need to predict.
label_to_id, id_to_label = ab.encode_labels(load_train())
assert len(label_to_id) == 906
print(f"{len(label_to_id)} classes encoded")

## 3. Tokenisation sanity check

`max_length=48` is fixed per `01_eda`'s question-length percentiles (mean 8.4 words, 99th
percentile 18) -- confirmed here on the fit split, not re-tuned. This truncation rate belongs in
report §2.2.

In [ ]:
from transformers import AutoTokenizer

MAX_LENGTH = 48
sanity_tokeniser = AutoTokenizer.from_pretrained("bert-base-uncased")
rate = ab.truncation_rate(fit["question"].tolist(), sanity_tokeniser, MAX_LENGTH)
print(f"truncation rate at max_length={MAX_LENGTH}: {rate:.4f}")

## 4. Train both encoders

`run_model_ablation` trains Bio_ClinicalBERT and `bert-base-uncased` with **identical**
hyperparameters, seed, and epochs -- the only thing that varies is the checkpoint. This is the
in-domain-pretraining claim in report §1.1: it has to be measured, not asserted. Bio_ClinicalBERT
is read out first as the primary run; the full ablation table follows.

`QUICK_SMOKE_TEST` cuts `NUM_EPOCHS`/`BATCH_SIZE` down for a fast local correctness check --
these are not the values to freeze.

In [ ]:
MODEL_NAMES = ["emilyalsentzer/Bio_ClinicalBERT", "bert-base-uncased"]

if QUICK_SMOKE_TEST:
    LEARNING_RATE = 2e-5
    BATCH_SIZE = 8
    NUM_EPOCHS = 1
else:
    LEARNING_RATE = 2e-5
    BATCH_SIZE = 16
    NUM_EPOCHS = 24

def report(model_name, row):
    short = model_name.split("/")[-1]
    print(
        f"{short} | epoch {row['epoch'] + 1}/{NUM_EPOCHS} | "
        f"train_loss {row['train_loss']:.4f} | val_loss {row['val_loss']:.4f} | "
        f"val_acc {row['val_accuracy']:.4f}"
    )

set_seed()
start = time.perf_counter()
ablation_summary, histories, ranked_by_epoch = ab.run_model_ablation(
    fit, val, label_to_id, MODEL_NAMES,
    lr=LEARNING_RATE, batch_size=BATCH_SIZE, epochs=NUM_EPOCHS, max_length=MAX_LENGTH, seed=44,
    on_epoch_end=report,
)
print(f"total wall clock for both encoders: {time.perf_counter() - start:.1f}s")
ablation_summary

In [ ]:
history_bioclinicalbert = histories["emilyalsentzer/Bio_ClinicalBERT"]
history_bertbase = histories["bert-base-uncased"]

history_bioclinicalbert.to_csv(ARTEFACTS_DIR / "arm2_history_bioclinicalbert.csv", index=False)
history_bertbase.to_csv(ARTEFACTS_DIR / "arm2_history_bertbase.csv", index=False)
ablation_summary.to_csv(ARTEFACTS_DIR / "arm2_model_ablation.csv", index=False)

print("Bio_ClinicalBERT (primary run):")
print(ablation_summary[ablation_summary.model_name == "emilyalsentzer/Bio_ClinicalBERT"].to_string(index=False))
print()
print("full ablation table:")
ablation_summary

## 5. F6 — training/validation loss curves

Report §4.2 reads the train/val gap as evidence of memorisation under ~10 examples per class, so
both curves are plotted for both encoders.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
for name, history in histories.items():
    short = name.split("/")[-1]
    ax.plot(history["epoch"], history["train_loss"], marker="o", label=f"{short} train")
    ax.plot(history["epoch"], history["val_loss"], marker="o", linestyle="--", label=f"{short} val")
ax.set_xlabel("epoch")
ax.set_ylabel("loss")
ax.set_title("Arm 2 — training/validation loss")
ax.legend()
fig.tight_layout()
fig.savefig(FIGURES_DIR / "fig6_loss_curves.png", dpi=150)
plt.show()

## 6. Selection

Epoch/checkpoint selection uses validation accuracy on the **standard hold-out only** -- Arm 2
does not use the shift-aware tie-break rule (that governs Arm 1 index-variant/scheme selection
only, per `CLAUDE.md`). Within-1-SE-prefer-fewest-epochs discipline applies throughout: SE ≈
3.5pp at n=200, so differences below ~7pp are noise.

In [ ]:
### 6a. Epoch selection, within each encoder
selected_bioclinicalbert = ab.select_best_epoch_within_one_se(history_bioclinicalbert, n_val=len(val))
selected_bertbase = ab.select_best_epoch_within_one_se(history_bertbase, n_val=len(val))

print("Bio_ClinicalBERT selected epoch (within 1 SE, fewest epochs):")
print(selected_bioclinicalbert)
print()
print("bert-base-uncased selected epoch (within 1 SE, fewest epochs):")
print(selected_bertbase)

selected_epoch_by_model = {
    "emilyalsentzer/Bio_ClinicalBERT": int(selected_bioclinicalbert["epoch"]),
    "bert-base-uncased": int(selected_bertbase["epoch"]),
}
print(f"\nselected epochs: {selected_epoch_by_model}")
print(f"budget was {NUM_EPOCHS} epochs -- both selections should sit strictly inside it")

In [ ]:
### 6b. Per-item validation predictions for BOTH encoders, then McNemar
# Taken from the rankings train_model captured during the ablation run above -- no retraining,
# so these are exactly the checkpoints whose accuracies §6a compared. Persisting both (not just
# the winner) is what makes the encoder ablation testable per-item: McNemar needs the runner-up's
# predictions, and report §3.2 requires a McNemar result for every system comparison.
SHORT_NAME = {"emilyalsentzer/Bio_ClinicalBERT": "bioclinicalbert", "bert-base-uncased": "bertbase"}

encoder_predictions = {}
for model_name, epoch in selected_epoch_by_model.items():
    ranked_at_epoch = ranked_by_epoch[model_name][epoch]
    frame = pd.DataFrame({"question": val["question"], "gold": val["disease"]})
    frame["pred"] = [r[0] for r in ranked_at_epoch]
    for i in range(5):
        frame[f"top_{i + 1}"] = [r[i] if i < len(r) else None for r in ranked_at_epoch]
    path = ARTEFACTS_DIR / f"arm2_val_predictions_{SHORT_NAME[model_name]}.csv"
    frame.to_csv(path, index=False)
    encoder_predictions[model_name] = frame
    accuracy = (frame["pred"] == frame["gold"]).mean()
    print(f"{SHORT_NAME[model_name]}: epoch {epoch}, accuracy {accuracy:.4f} -> {path.name}")

mcnemar = evaluate.mcnemar_exact(
    encoder_predictions["emilyalsentzer/Bio_ClinicalBERT"]["pred"].tolist(),
    encoder_predictions["bert-base-uncased"]["pred"].tolist(),
    val["disease"].tolist(),
)
print("\nMcNemar exact -- a = Bio_ClinicalBERT, b = bert-base-uncased,")
print("each at its own selected epoch, over the same validation items:")
for key, value in mcnemar.items():
    print(f"  {key}: {value}")

pd.DataFrame([{"system_a": "emilyalsentzer/Bio_ClinicalBERT", "system_b": "bert-base-uncased", **mcnemar}]).to_csv(
    ARTEFACTS_DIR / "arm2_encoder_mcnemar.csv", index=False
)

In [ ]:
### 6c. Encoder choice
# Pre-registered rule, recorded in CLAUDE.md (2026-08-17): if McNemar does not separate the two
# encoders on the standard hold-out, the comparison is reported as unresolved and the in-domain
# clinical encoder is kept on prior grounds -- a declared prior, not a reading of the numbers.
# Only if the test resolves the comparison does measured accuracy decide.
ALPHA = 0.05

if mcnemar["p_value"] >= ALPHA:
    print(f"McNemar p = {mcnemar['p_value']:.4f} >= alpha = {ALPHA}")
    print("Unresolved -- the hold-out does not separate these encoders. Not declaring a winner.")
    print("Applying the pre-registered encoder tie-break: keep the in-domain clinical encoder.")
    print("NOTE: this selects Bio_ClinicalBERT regardless of which encoder scored higher.")
    selected_model_name = "emilyalsentzer/Bio_ClinicalBERT"
    selected_row = selected_bioclinicalbert
else:
    print(f"McNemar p = {mcnemar['p_value']:.4f} < alpha = {ALPHA}: the difference is reliable.")
    print("Tie-break does not fire; measured accuracy decides.")
    higher_is_bioclinical = selected_bioclinicalbert["val_accuracy"] > selected_bertbase["val_accuracy"]
    selected_model_name = "emilyalsentzer/Bio_ClinicalBERT" if higher_is_bioclinical else "bert-base-uncased"
    selected_row = selected_bioclinicalbert if higher_is_bioclinical else selected_bertbase

print(f"\nselected model: {selected_model_name}")
print(f"selected epoch: {int(selected_row['epoch'])} (val_accuracy {selected_row['val_accuracy']:.4f})")

## 7. Validation predictions for the selected model

Persisted with per-item top-5 predictions, aligned to validation order -- `05_results` and the
error analysis in §3.3 both need this, and McNemar against Arm 1 needs per-item predictions
aligned to the same validation set.

In [ ]:
from transformers import AutoModelForSequenceClassification

# Re-fit the selected model at its selected epoch count to obtain the state dict used for the
# persisted predictions (run_model_ablation already trained it above; retrain isolated to avoid
# holding every encoder's full state_dict in memory across the notebook).
set_seed()
selected_epochs = int(selected_row["epoch"]) + 1
selected_state, _, _ = ab.train_model(
    fit, val, label_to_id, selected_model_name,
    lr=LEARNING_RATE, batch_size=BATCH_SIZE, epochs=selected_epochs, max_length=MAX_LENGTH, seed=44,
)
final_model = AutoModelForSequenceClassification.from_pretrained(
    selected_model_name, num_labels=len(label_to_id)
)
final_model.load_state_dict(selected_state)
final_model = final_model.to(device)
final_tokeniser = AutoTokenizer.from_pretrained(selected_model_name)

# top_k=None ranks all 906 labels. Arm 1 ranks every class, so a top-5 ranking here would make
# Arm 2's MRR an MRR@5 and the two arms' MRR columns non-comparable.
ranked = ab.predict_ranked(
    final_model, final_tokeniser, val["question"].tolist(), id_to_label,
    max_length=MAX_LENGTH, batch_size=BATCH_SIZE, device=device, top_k=None,
)
scores = evaluate.score_ranked(ranked, val["disease"].tolist())
print(scores)

# The refit replays the same seed and data, so its top-1 should reproduce what §6b captured for
# this encoder during the ablation run. Printed rather than asserted: cuDNN kernel selection is
# not bit-deterministic, so a small disagreement is a hardware artefact, not a logic error. A
# large one would mean the persisted predictions and the compared checkpoint have diverged.
captured_top1 = encoder_predictions[selected_model_name]["pred"].tolist()
refit_top1 = [r[0] for r in ranked]
agreement = sum(a == b for a, b in zip(captured_top1, refit_top1)) / len(val)
print(f"refit vs §6b captured top-1 agreement: {agreement:.4f}")

val_predictions = pd.DataFrame({"question": val["question"], "gold": val["disease"]})
val_predictions["pred"] = [r[0] for r in ranked]
for i in range(5):
    val_predictions[f"top_{i + 1}"] = [r[i] if i < len(r) else None for r in ranked]
val_predictions.to_csv(ARTEFACTS_DIR / "arm2_val_predictions.csv", index=False)
val_predictions.head()

pd.DataFrame(
    [{**scores, "model_name": selected_model_name, "epoch": selected_epochs - 1, "refit_agreement": agreement}]
).to_csv(ARTEFACTS_DIR / "arm2_val_metrics.csv", index=False)

if DRIVE_DIR is not None:
    try:
        checkpoint_path = DRIVE_DIR / "arm2_selected_model.pt"
        torch.save(
            {
                "state_dict": selected_state,
                "model_name": selected_model_name,
                "num_epochs": selected_epochs,
                "max_length": MAX_LENGTH,
                "label_to_id": label_to_id,
            },
            checkpoint_path,
        )
        size_mb = checkpoint_path.stat().st_size / 1e6
        print(f"checkpoint saved to {checkpoint_path} ({size_mb:.1f} MB)")
    except Exception as e:
        print(f"checkpoint save failed: {e}")
else:
    print("checkpoint NOT saved -- 05_results will need to retrain")

## 8. Freeze Arm 2 hyperparameters into `config.py`

Values printed here, then copied into `src/amlh/config.py`'s `HYPERPARAMETERS` Arm 2 fields as a
follow-up source edit -- matching how Arm 1's hyperparameters were frozen. **Only valid when
`QUICK_SMOKE_TEST = False`** (a real Colab T4 run) -- the smoke-test run above exists to prove
the notebook's logic is correct, not to supply frozen values.

In [ ]:
frozen_arm2_hyperparameters = {
    "bert_model_name": selected_model_name,
    "max_length": MAX_LENGTH,
    "learning_rate": LEARNING_RATE,
    "batch_size": BATCH_SIZE,
    "num_epochs": selected_epochs,
}
print(f"QUICK_SMOKE_TEST = {QUICK_SMOKE_TEST}")
if QUICK_SMOKE_TEST:
    print("NOT VALID TO FREEZE -- re-run with QUICK_SMOKE_TEST=False on Colab first.")
print(frozen_arm2_hyperparameters)

In [ ]:
import zipfile

arm2_outputs = [
    ARTEFACTS_DIR / "arm2_history_bioclinicalbert.csv",
    ARTEFACTS_DIR / "arm2_history_bertbase.csv",
    ARTEFACTS_DIR / "arm2_model_ablation.csv",
    ARTEFACTS_DIR / "arm2_val_predictions.csv",
    ARTEFACTS_DIR / "arm2_val_predictions_bioclinicalbert.csv",
    ARTEFACTS_DIR / "arm2_val_predictions_bertbase.csv",
    ARTEFACTS_DIR / "arm2_encoder_mcnemar.csv",
    ARTEFACTS_DIR / "arm2_val_metrics.csv",
    FIGURES_DIR / "fig6_loss_curves.png",
]

try:
    from google.colab import files
    on_colab = True
except ImportError:
    on_colab = False

export_path = ARTEFACTS_DIR / "arm2_colab_outputs.zip"
export_path.parent.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(export_path, "w") as zf:
    for path in arm2_outputs:
        if path.exists():
            prefix = "figures" if path.parent == FIGURES_DIR else "artefacts"
            zf.write(path, arcname=f"{prefix}/{path.name}")
        else:
            print(f"missing, skipped: {path}")

print(f"wrote {export_path}")
if on_colab:
    files.download(str(export_path))
    print("downloaded to Colab on path: ", str(export_path))
else:
    print("not running on Colab -- files are already local, skipping download.")